In [1]:
import os
import json
import pickle
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd


from tqdm import tqdm
from xgboost import XGBRanker
from sentence_transformers import SentenceTransformer


c:\Users\franc\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)


def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)


def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0


def hit_score_at_k(rec_k, rel_set):
    cant_relevantes = set(rec_k).intersection(set(rel_set))
    return len(cant_relevantes)


def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0


def diversity_at_k(rec_k, info_videojuegos):
    generos_total = set()
    for app_id in rec_k:
        for genero in info_videojuegos[app_id]:
            #print(genero)
            generos_total.add(genero)
    if not generos_total:
        return 0
    return len(generos_total)


def f1_at_k(rec_k, rel_set):
    if len(rec_k) == 0 or len(rel_set) == 0:
        return 0.0

    p = precision_at_k(rec_k, rel_set)
    r = recall_at_k(rec_k, rel_set)

    if (p + r) <= 0:
        return 0.0

    return 2 * p * r / (p + r)


In [3]:

base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "..", "data", "split")

ruta_train = os.path.join(data_dir, "train_split.csv")
ruta_test = os.path.join(data_dir, "test_split.csv")
ruta_val = os.path.join(data_dir, "val_split.csv")

ruta_metadata = os.path.join("games_metadata.json")
ruta_imagenes_url = os.path.join("steam_media_data.csv")

train_set = pd.read_csv(ruta_train)
test_set = pd.read_csv(ruta_test)

train_set["hours"] = np.log1p(train_set["hours"])
test_set["hours"] = np.log1p(test_set["hours"])

regla_rating = {True: 1, False: 0}
train_set['rating'] = train_set['is_recommended'].map(regla_rating)
test_set['rating'] = test_set['is_recommended'].map(regla_rating)

ratings_ = test_set[test_set["rating"] == 1]
items_relevantes = test_set.groupby("user_id")["app_id"].apply(list).to_dict()


In [4]:
final_dict = {}
info_videojuegos = defaultdict(list)
set_tags = set()

with open(ruta_metadata, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        app_id = obj["app_id"]
        final_dict[app_id] = str(obj["description"])
        info_videojuegos[app_id].extend(obj["tags"])

        for tag in obj["tags"]:
            set_tags.add(tag)

# Lista total de app_id presentes en train y test
app_ids_total = train_set["app_id"].tolist()
app_ids_total.extend(test_set["app_id"].tolist())
app_ids_total = list(set(app_ids_total))
len(app_ids_total)


2880

In [5]:
info_videojuegos

defaultdict(list,
            {13500: ['Action',
              'Adventure',
              'Parkour',
              'Third Person',
              'Great Soundtrack',
              'Singleplayer',
              'Platformer',
              'Time Travel',
              'Atmospheric',
              'Classic',
              'Hack and Slash',
              'Time Manipulation',
              'Gore',
              'Fantasy',
              'Story Rich',
              'Dark',
              'Open World',
              'Controller',
              'Dark Fantasy',
              'Puzzle'],
             22364: ['Action'],
             113020: ['Co-op',
              'Stealth',
              'Indie',
              'Heist',
              'Local Co-Op',
              'Strategy',
              'Online Co-Op',
              'Top-Down',
              'Action',
              'Multiplayer',
              'Crime',
              'Casual',
              'Great Soundtrack',
              'Adventure',
             

In [6]:
#img_data = pd.read_csv(ruta_imagenes_url)
#img_data = img_data[["steam_appid", "header_image"]]
#img_data = img_data[img_data["steam_appid"].isin(app_ids_total)]
#img_data.head()


In [7]:


# import requests
# from io import BytesIO
# from PIL import Image
# from sentence_transformers import SentenceTransformer
#
# model_img = SentenceTransformer("clip-ViT-B-32")
#
# embeddings_dict = {}
#
# for _, row in tqdm(img_data.iterrows(), total=len(img_data), desc="Procesando imágenes"):
#     image_id = int(row["steam_appid"])
#     url = row["header_image"]
#
#     try:
#         # Descargar imagen
#         response = requests.get(url, timeout=10)
#         response.raise_for_status()
#
#         # Abrir como PIL
#         img = Image.open(BytesIO(response.content)).convert("RGB")
#
#         # Embedding
#         emb = model_img.encode(img)
#
#         embeddings_dict[image_id] = emb
#
#     except Exception as e:
#         print(f"⚠️ Error con ID {image_id}, URL {url}: {e}")
#

# with open("embeddings_dict.pkl", "wb") as f:
#     pickle.dump(embeddings_dict, f)


In [8]:
with open("embeddings_dict.pkl", "rb") as f:
    embeddings_dict = pickle.load(f)

embeddings_dict = {int(k): np.array(v) for k, v in embeddings_dict.items()}

train_set["app_id"] = train_set["app_id"].astype(int)
test_set["app_id"] = test_set["app_id"].astype(int)


In [9]:
descripciones = list(final_dict.values())
keys_app_id = list(final_dict.keys())

train_set = train_set.sort_values("user_id").reset_index(drop=True)
test_set = test_set.sort_values("user_id").reset_index(drop=True)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings_text = model.encode(descripciones, show_progress_bar=True)

dict_transformados = {int(i): j for i, j in zip(keys_app_id, embeddings_text)}

train_set["descripciones"] = train_set["app_id"].map(dict_transformados)
test_set["descripciones"] = test_set["app_id"].map(dict_transformados)


Batches: 100%|██████████| 1590/1590 [00:13<00:00, 121.24it/s]


In [10]:
img_dim = next(iter(embeddings_dict.values())).shape[0]
zero_img = np.zeros(img_dim, dtype=np.float32)

train_set["emb_img"] = train_set["app_id"].map(embeddings_dict).apply(
    lambda x: x if isinstance(x, np.ndarray) else zero_img
)
test_set["emb_img"] = test_set["app_id"].map(embeddings_dict).apply(
    lambda x: x if isinstance(x, np.ndarray) else zero_img
)

columnas_importantes = ["user_id", "app_id", "hours", "descripciones", "emb_img"]
train_set = train_set[columnas_importantes]
test_set = test_set[columnas_importantes]

train_set.head()


,user_id,app_id,hours,descripciones,emb_img
0,731,322330,4.226834,"[-0.1188384, 0.04829871, -0.0025480792, -0.011...","[-0.1327441, 0.27538148, -0.24782176, 0.193519..."
1,731,394360,6.003146,"[-0.1188384, 0.04829871, -0.0025480792, -0.011...","[-0.2154008, -0.29375905, -0.28681642, -0.2764..."
2,731,4700,6.528689,"[0.0742551, 0.027489569, 0.019467622, -0.02517...","[-0.64909875, 0.1372838, 0.23721811, 0.3455800..."
3,731,433340,3.505557,"[-0.1188384, 0.04829871, -0.0025480792, -0.011...","[-0.5361488, 0.3972314, 0.09718392, 0.03582586..."
4,3128,548430,4.672829,"[-0.1188384, 0.04829871, -0.0025480792, -0.011...","[0.17213973, 0.31455576, -0.12051097, 0.369043..."


In [11]:
columnas_train, columnas_predict = ["user_id", "app_id", "descripciones", "emb_img"], ["hours"]
X_train_df, y_train = train_set[columnas_train], train_set[columnas_predict]

numerical_features_train = X_train_df[["user_id", "app_id"]].values

descripciones_dense_train = np.vstack(X_train_df["descripciones"].to_numpy())

img_emb_train = np.vstack(X_train_df["emb_img"].to_numpy())

X_train = np.hstack((numerical_features_train,
                     descripciones_dense_train,
                     img_emb_train))

group_train = train_set.groupby("user_id").size().tolist()
group_test = test_set.groupby("user_id").size().tolist() 

X_train.shape, len(group_train)


((63905, 898), 9906)

In [12]:
ranker = XGBRanker(
    objective="rank:pairwise",
    learning_rate=0.1,
    n_estimators=200,
    max_depth=4,
    subsample=1,
    colsample_bytree=1,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)


,objective,'rank:pairwise'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
recomendaciones = defaultdict(list)

test_set_user_uniques = test_set["user_id"].unique().tolist()
test_set_app_uniques = test_set["app_id"].unique().tolist()
app_ids_array = np.array(list(test_set_app_uniques))

for user_id in tqdm(test_set_user_uniques, desc="Prediciendo por usuario"):
    app_ids = app_ids_array
    user_ids = np.full_like(app_ids, fill_value=user_id)

    numerical_feats = np.stack([user_ids, app_ids], axis=1)

    desc_feats = np.stack(
        [dict_transformados[int(app_id)] for app_id in app_ids]
    )

    img_feats = np.stack(
        [embeddings_dict.get(int(app_id), zero_img) for app_id in app_ids]
    )

    X_user = np.hstack((numerical_feats, desc_feats, img_feats))

    scores = ranker.predict(X_user)
    recomendaciones[user_id] = list(zip(app_ids, scores))

recomendaciones_def = defaultdict(list)
for usuario, lista_recs in recomendaciones.items():
    recomendaciones_ord = sorted(lista_recs, key=lambda x: x[1], reverse=True)
    recomendaciones_ord = [i[0] for i in recomendaciones_ord]
    recomendaciones_def[usuario] = recomendaciones_ord


Prediciendo por usuario: 100%|██████████| 9906/9906 [02:38<00:00, 62.56it/s]


In [15]:
precision_list = list()
recall_list = list()
ndcg_list = list()
f1_list = list()
hitrate_list = list()
map10_list = list()
diversity_list = list()

for usuario, recomendaciones_usuario in recomendaciones_def.items():
    items_rel_usuario = items_relevantes[usuario]
    recomendaciones_10 = recomendaciones_usuario[:10]
    
    precision_usuario = precision_at_k(recomendaciones_10, items_rel_usuario)
    recall_usuario = recall_at_k(recomendaciones_10, items_rel_usuario)
    ndcg_usuario = ndcg_at_k(recomendaciones_10, items_rel_usuario)
    f1_usuario = f1_at_k(recomendaciones_10, items_rel_usuario)
    hitrate_usuario = hit_score_at_k(recomendaciones_10, items_rel_usuario)
    map10_usuario = map_at_k(recomendaciones_10, items_rel_usuario)
    diversity_usuario = diversity_at_k(recomendaciones_10, info_videojuegos)
    
    precision_list.append(precision_usuario)
    recall_list.append(recall_usuario)
    ndcg_list.append(ndcg_usuario)
    f1_list.append(f1_usuario)
    hitrate_list.append(hitrate_usuario)
    map10_list.append(map10_usuario)
    diversity_list.append(diversity_usuario)

In [16]:
print(f"Precision@10: {np.mean(precision_list):.4f}")
print(f"Recall@10: {np.mean(recall_list):.4f}")
print(f"F1-Score@10: {np.mean(f1_list):.4f}")
print(f"Hit Score@10: {np.mean(hitrate_list):.4f}")
print(f"nDCG@10: {np.mean(ndcg_list):.4f}")
print(f"MAP@10: {np.mean(map10_list):.4f}")
print(f"Diversity: {np.mean(diversity_list):.4f}")

Precision@10: 0.0023
Recall@10: 0.0204
F1-Score@10: 0.0041
Hit Score@10: 0.0229
nDCG@10: 0.0089
MAP@10: 0.0052
Diversity: 0.0000


In [27]:
contador = 0
for usuario, recomendaciones_usuario in (recomendaciones_def.items()):
    items_rel_usuario = items_relevantes[usuario]
    recomendaciones_10 = recomendaciones_usuario[:10]

    print(recomendaciones_10)
    print(info_videojuegos[recomendaciones_10[0]])

    contador += 1
    if contador > 30:
        break


[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570), np.int64(440), np.int64(250900)]
[]
[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570), np.int64(440), np.int64(250900)]
[]
[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570), np.int64(440), np.int64(250900)]
[]
[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570), np.int64(440), np.int64(250900)]
[]
[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570), np.int64(440), np.int64(250900)]
[]
[np.int64(252490), np.int64(230410), np.int64(730), np.int64(359550), np.int64(394360), np.int64(271590), np.int64(4000), np.int64(570)

In [25]:
info_videojuegos[230410]

[]